In [84]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
 
from sklearn.preprocessing import PowerTransformer


In [85]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [86]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [87]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [88]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [89]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [90]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [91]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [92]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [93]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [94]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [95]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [96]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [97]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [98]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [99]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [100]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [101]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"shape_Elongation",
"shape_MajorAxisLength",
"shape_Sphericity",
"shape_SurfaceVolumeRatio",
"shape_Flatness",
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16"
]

In [102]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [103]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Yeo-Johnson Transformation

In [104]:
# Copy the original X for later 
original_X = X.copy()

In [105]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)

# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [106]:
X_new

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16
0,0.600926,42.073251,0.761164,0.251218,0.535140,0.029425
1,0.841579,24.613845,0.697049,0.489853,0.367109,0.037915
2,0.772821,48.030294,0.565792,0.278467,0.597785,0.008009
3,0.847727,25.589900,0.684364,0.474018,0.405730,0.018398
4,0.831483,34.684750,0.503142,0.563135,0.442406,0.013051
...,...,...,...,...,...,...
134,0.680294,33.069705,0.742102,0.322021,0.523608,0.014938
135,0.758193,41.043692,0.722918,0.227705,0.735524,0.011441
136,0.770113,36.618802,0.652963,0.298398,0.648063,0.020431
137,0.628897,45.870392,0.724255,0.252893,0.492193,0.019663


In [107]:
X_new_std

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16
0,-0.735252,0.072022,1.124852,-0.476515,0.067055,1.294944
1,1.029207,-1.421627,0.161153,1.471846,-1.298903,1.828126
2,0.480839,0.444135,-1.401990,-0.164875,0.581443,-1.587057
3,1.080021,-1.314128,-0.012817,1.384484,-0.986786,0.180452
4,0.946400,-0.468644,-1.982774,1.825476,-0.689355,-0.617219
...,...,...,...,...,...,...
134,-0.200607,-0.601731,0.823106,0.274152,-0.027337,-0.311625
135,0.368838,0.002519,0.532655,-0.771682,1.721586,-0.901174
136,0.459982,-0.316986,-0.421265,0.044444,0.996190,0.433122
137,-0.551976,0.314722,0.552472,-0.456457,-0.284014,0.340615


In [108]:
MAASTRO_new 

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16
0,0.765178,50.002093,0.668072,0.215184,0.610062,0.010495
1,0.776540,41.753334,0.669961,0.276092,0.504616,0.035018
2,0.697164,44.375483,0.624081,0.298887,0.478604,0.012819
3,0.574636,46.115989,0.577624,0.361096,0.446059,0.029974
4,0.633419,54.394967,0.630933,0.251519,0.480378,0.013458
...,...,...,...,...,...,...
94,0.882411,34.218615,0.671754,0.307574,0.577884,0.039092
95,0.535802,51.046869,0.632189,0.289922,0.455642,0.015831
96,0.716610,50.417953,0.645548,0.228184,0.631485,0.011031
97,0.665145,44.901412,0.727488,0.223872,0.628338,0.019801


In [109]:
MAASTRO_new_std

,shape_Elongation,shape_MajorAxisLength,shape_Sphericity,shape_SurfaceVolumeRatio,shape_Flatness,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16
0,0.422117,0.557388,-0.228627,-0.939751,0.682565,-1.078945
1,0.509566,0.050605,-0.204032,-0.190837,-0.182599,1.669991
2,-0.081096,0.221607,-0.770221,0.049387,-0.394828,-0.656860
3,-0.902489,0.329735,-1.281223,0.615142,-0.659672,1.336360
4,-0.521824,0.794695,-0.689674,-0.472901,-0.380366,-0.548839
...,...,...,...,...,...,...
94,1.372275,-0.506427,-0.180591,0.135942,0.417742,1.886364
95,-1.140744,0.615633,-0.674760,-0.042764,-0.581768,-0.176666
96,0.059266,0.580714,-0.513250,-0.765405,0.859251,-0.977202
97,-0.306155,0.254712,0.600661,-0.822284,0.833281,0.357476


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [110]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 17:26:25,411] A new study created in memory with name: no-name-8e4d94b6-7934-4852-adf3-a7e93ec7ccb7


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7248062015503876
Fold 3 C-index: 0.6297872340425532
Fold 4 C-index: 0.714828897338403


[I 2024-04-15 17:26:25,806] A new study created in memory with name: no-name-4d56ea40-d88b-4927-8e1c-207c1424d40d


Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:25,796] Trial 0 finished with value: 0.6751603929238369 and parameters: {}. Best is trial 0 with value: 0.6751603929238369.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6751603929238369], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 25, 453372), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 25, 796514), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6751603929238369


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2518617933470256
Fold 2 IBS: 0.18864185825386184
Fold 3 IBS: 0.2305846597429618
Fold 4 IBS: 0.2244901345824116
Fold 5 IBS: 0.20899645104594913
[I 2024-04-15 17:26:26,281] Trial 0 finished with value: 0.220914979394442 and parameters: {}. Best is trial 0 with value: 0.220914979394442.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.220914979394442], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 25, 857949), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 26, 281442), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.220914979394442


In [111]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [112]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.675
train_ibs:  0.221


#### Test

In [113]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [114]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.498
IBS score: 0.281


In [115]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [116]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [117]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:26:26,508] A new study created in memory with name: no-name-14ad2f4e-8db6-452e-866f-61ec523c9e25


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.603585657370518
Fold 2 C-index: 0.6957364341085271
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.7262357414448669


[I 2024-04-15 17:26:26,969] A new study created in memory with name: no-name-1dfda8aa-6aae-4f71-be94-a85ccea5eca9


Fold 5 C-index: 0.6673819742489271
[I 2024-04-15 17:26:26,960] Trial 0 finished with value: 0.6492262593069082 and parameters: {}. Best is trial 0 with value: 0.6492262593069082.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6492262593069082], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 26, 543298), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 26, 959929), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6492262593069082


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709252196547
Fold 2 IBS: 0.2320398736627342
Fold 3 IBS: 0.22898186761465514
Fold 4 IBS: 0.24197475887753253
Fold 5 IBS: 0.22939558519938744
[I 2024-04-15 17:26:27,310] Trial 0 finished with value: 0.23592783557525493 and parameters: {}. Best is trial 0 with value: 0.23592783557525493.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592783557525493], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 27, 8548), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 27, 309933), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592783557525493


In [118]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [119]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.649
train_ibs:  0.236


#### Test

In [120]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [121]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [122]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [123]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:26:27,594] A new study created in memory with name: no-name-030724bd-ffba-4282-80e6-0ff409784d7b


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.6212765957446809


[I 2024-04-15 17:26:28,145] A new study created in memory with name: no-name-337f0fbb-2a6c-499e-b5d6-fc4168cf5670


Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:28,125] Trial 0 finished with value: 0.6734435277395772 and parameters: {}. Best is trial 0 with value: 0.6734435277395772.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6734435277395772], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 27, 628110), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 28, 125497), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6734435277395772


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2515309590238497
Fold 2 IBS: 0.1885602195226402
Fold 3 IBS: 0.2296957276388316
Fold 4 IBS: 0.22310595642466954
Fold 5 IBS: 0.20623729181941608
[I 2024-04-15 17:26:28,851] Trial 0 finished with value: 0.21982603088588143 and parameters: {}. Best is trial 0 with value: 0.21982603088588143.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21982603088588143], datetime_start=datetime.datetime(2024, 4, 15, 17, 26, 28, 188966), datetime_complete=datetime.datetime(2024, 4, 15, 17, 26, 28, 851153), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21982603088588143


In [124]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [125]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.673
train_ibs:  0.22


#### Test

In [126]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [127]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.497


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.279


In [128]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [129]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:26:29,155] A new study created in memory with name: no-name-aca920ca-0277-485b-89af-976529085ac4


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:29,651] Trial 0 finished with value: 0.6742945915693643 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6742945915693643.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:30,143] Trial 1 finished with value: 0.6750914043183683 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.6750914043183683.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:30,676] Trial 2 finished with value: 0.6750914043183683 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:44,631] Trial 24 finished with value: 0.6750914043183683 and parameters: {'l1_ratio': 0.21623254971460304}. Best is trial 12 with value: 0.6767935319779428.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.6340425531914894
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:45,252] Trial 25 finished with value: 0.6767935319779428 and parameters: {'l1_ratio': 0.0853699617771228}. Best is trial 12 with value: 0.6767935319779428.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:45,814] Trial 26 finished with value: 0.6606233192119854 and parameters: {'l1_ratio': 0.059959030692121

Fold 1 C-index: 0.5896414342629482
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:26:59,338] Trial 48 finished with value: 0.6742945915693643 and parameters: {'l1_ratio': 0.5683872043393509}. Best is trial 12 with value: 0.6767935319779428.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.6340425531914894
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:27:00,007] Trial 49 finished with value: 0.6767935319779428 and parameters: {'l1_ratio': 0.08054512980432653}. Best is trial 12 with value: 0.6767935319779428.
Fold 1 C-index: 0.5936254980079682
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.625531914893617
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7167381974248928
[I 2024-04-15 17:27:00,708] Trial 50 finished with value: 0.6750914043183683 and parameters: {'l1_ratio': 0.1849235252706911

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.2515901749755985
Fold 2 IBS: 0.18858261003685164
Fold 3 IBS: 0.2292920841299743
Fold 4 IBS: 0.22309648532344112
Fold 5 IBS: 0.20605265728959898
[I 2024-04-15 17:27:25,275] Trial 0 finished with value: 0.21972280235109293 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.21972280235109293.
Fold 1 IBS: 0.2516047274950201
Fold 2 IBS: 0.18861654294846003
Fold 3 IBS: 0.22828249837494724
Fold 4 IBS: 0.22300001654003024
Fold 5 IBS: 0.20725123192572797
[I 2024-04-15 17:27:25,831] Trial 1 finished with value: 0.21975100345683712 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.21972280235109293.
Fold 1 IBS: 0.2515771273750895
Fold 2 IBS: 0.18856022235091993
Fold 3 IBS: 0.22808587657282242
Fold 4 IBS: 0.22299628787333223
Fold 5 IBS: 0.20718466250448006
[I 2024-04-15 17:27:26,313] Trial 2 finished with value: 0.21968083533532884 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2196808353353

Fold 1 IBS: 0.2517558250797045
Fold 2 IBS: 0.22831284027786944
Fold 3 IBS: 0.2288517859642901
Fold 4 IBS: 0.2376370340824302
Fold 5 IBS: 0.22662832135964203
[I 2024-04-15 17:27:38,731] Trial 25 finished with value: 0.23463716135278725 and parameters: {'l1_ratio': 0.03981317955666948}. Best is trial 12 with value: 0.21954497356693184.
Fold 1 IBS: 0.25158208088435086
Fold 2 IBS: 0.18851476319883065
Fold 3 IBS: 0.22786796404123397
Fold 4 IBS: 0.22300557477574828
Fold 5 IBS: 0.2071728242046949
[I 2024-04-15 17:27:39,571] Trial 26 finished with value: 0.21962864142097174 and parameters: {'l1_ratio': 0.18783928730207888}. Best is trial 12 with value: 0.21954497356693184.
Fold 1 IBS: 0.25160373191034385
Fold 2 IBS: 0.18851369676515642
Fold 3 IBS: 0.2273527594397358
Fold 4 IBS: 0.22315496471016386
Fold 5 IBS: 0.2070486022039014
[I 2024-04-15 17:27:40,431] Trial 27 finished with value: 0.21953475100586023 and parameters: {'l1_ratio': 0.07505393448412474}. Best is trial 27 with value: 0.21953475

Fold 1 IBS: 0.25157283476309095
Fold 2 IBS: 0.188518438507461
Fold 3 IBS: 0.2274894350667155
Fold 4 IBS: 0.2231905141924281
Fold 5 IBS: 0.20711743659256523
[I 2024-04-15 17:27:55,448] Trial 50 finished with value: 0.21957773182445214 and parameters: {'l1_ratio': 0.10438246262749007}. Best is trial 27 with value: 0.21953475100586023.
Fold 1 IBS: 0.25161313178751893
Fold 2 IBS: 0.1885009692084376
Fold 3 IBS: 0.2273845031216082
Fold 4 IBS: 0.2231742041630635
Fold 5 IBS: 0.20707753492010803
[I 2024-04-15 17:27:56,131] Trial 51 finished with value: 0.2195500686401472 and parameters: {'l1_ratio': 0.07683940226987503}. Best is trial 27 with value: 0.21953475100586023.
Fold 1 IBS: 0.24559521233044804
Fold 2 IBS: 0.2297823454798514
Fold 3 IBS: 0.22889621945120459
Fold 4 IBS: 0.2393529615265275
Fold 5 IBS: 0.2277379394307792
[I 2024-04-15 17:27:56,544] Trial 52 finished with value: 0.23427293564376214 and parameters: {'l1_ratio': 0.022665207963687185}. Best is trial 27 with value: 0.219534751005

Fold 1 IBS: 0.2516216316778847
Fold 2 IBS: 0.18848965387137187
Fold 3 IBS: 0.22762467780633341
Fold 4 IBS: 0.22308259673228695
Fold 5 IBS: 0.20709200183655982
[I 2024-04-15 17:28:10,551] Trial 75 finished with value: 0.21958211238488734 and parameters: {'l1_ratio': 0.12295487228655186}. Best is trial 27 with value: 0.21953475100586023.
Fold 1 IBS: 0.25156666498020597
Fold 2 IBS: 0.1885402793435447
Fold 3 IBS: 0.22800189108248853
Fold 4 IBS: 0.2230557257181538
Fold 5 IBS: 0.2071577454926291
[I 2024-04-15 17:28:11,155] Trial 76 finished with value: 0.2196644613234044 and parameters: {'l1_ratio': 0.20331332808738323}. Best is trial 27 with value: 0.21953475100586023.
Fold 1 IBS: 0.25161289049064667
Fold 2 IBS: 0.1885199340137061
Fold 3 IBS: 0.2273776537442294
Fold 4 IBS: 0.22323460298163733
Fold 5 IBS: 0.20706055936917897
[I 2024-04-15 17:28:11,768] Trial 77 finished with value: 0.21956112811987966 and parameters: {'l1_ratio': 0.08299419528943816}. Best is trial 27 with value: 0.219534751

In [130]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [131]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.677
train_ibs:  0.22


#### Test

In [132]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [133]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.0873928389796849)

test_cindex : 0.497


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07505393448412474)

test_ibs:  0.28


In [134]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [135]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 17:28:25,738] A new study created in memory with name: no-name-3f7e6d6c-4735-40bf-970e-d7be9d3a4e6a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6892430278884463
Fold 2 C-index: 0.7984496124031008
Fold 3 C-index: 0.6297872340425532
Fold 4 C-index: 0.7623574144486692
Fold 5 C-index: 0.6738197424892703
[I 2024-04-15 17:28:29,597] Trial 0 finished with value: 0.710731406254408 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.710731406254408.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.6510638297872341
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.703862660944206
[I 2024-04-15 17:28:32,489] Trial 1 finished with value: 0.7115081949165625 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'ma

Fold 2 C-index: 0.7751937984496124
Fold 3 C-index: 0.7531914893617021
Fold 4 C-index: 0.8326996197718631
Fold 5 C-index: 0.7982832618025751
[I 2024-04-15 17:29:08,230] Trial 15 finished with value: 0.7621525183393019 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 16, 'max_depth': 1, 'n_estimators': 491, 'oob_score': True, 'max_samples': 0.9914273801490951, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.08807762154599924, 'warm_start': True}. Best is trial 14 with value: 0.7644780997346508.
Fold 1 C-index: 0.6633466135458167
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.8155893536121673
Fold 5 C-index: 0.7982832618025751
[I 2024-04-15 17:29:11,245] Trial 16 finished with value: 0.7730522904564694 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 2, 'min_samples_leaf': 18, 'max_depth': 1, 'n_estimators': 499, 'oob_score': True, 'max_samples': 0.9631819737604822, 'max_features': 'log2', 'min_wei

Fold 1 C-index: 0.647410358565737
Fold 2 C-index: 0.8178294573643411
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.8193916349809885
Fold 5 C-index: 0.7939914163090128
[I 2024-04-15 17:29:27,032] Trial 30 finished with value: 0.7672139351461436 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 18, 'max_depth': 5, 'n_estimators': 125, 'oob_score': True, 'max_samples': 0.6865450555598549, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.2124753462487393, 'warm_start': True}. Best is trial 19 with value: 0.779878642525784.
Fold 1 C-index: 0.6533864541832669
Fold 2 C-index: 0.8255813953488372
Fold 3 C-index: 0.7702127659574468
Fold 4 C-index: 0.8326996197718631
Fold 5 C-index: 0.7939914163090128
[I 2024-04-15 17:29:28,656] Trial 31 finished with value: 0.7751743303140853 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 8, 'min_samples_leaf': 20, 'max_depth': 9, 'n_estimators': 346, 'oob_score': True, 'max_samples': 0.8836899167485477, 

...

Fold 1 IBS: 0.24607569530270887
Fold 2 IBS: 0.2322534676033703
Fold 3 IBS: 0.23000152640707228
Fold 4 IBS: 0.24129488323806544
Fold 5 IBS: 0.23027107814262227
[I 2024-04-15 17:33:39,022] Trial 32 finished with value: 0.23597933013876782 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 11, 'n_estimators': 390, 'oob_score': False, 'max_samples': 0.2693933566356703, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.16572038773562736}. Best is trial 28 with value: 0.2039609151270521.
Fold 1 IBS: 0.25666234269640353
Fold 2 IBS: 0.1921775892275604
Fold 3 IBS: 0.24131762851690544
Fold 4 IBS: 0.20035119005884414
Fold 5 IBS: 0.20538433727508362
[I 2024-04-15 17:33:43,097] Trial 33 finished with value: 0.21917861755495943 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 6, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 320, 'oob_score': False, 'max_samples': 0.7422917748630138, 'max_features': None, 'min_weight_fraction_l

Fold 1 IBS: 0.2469763086874542
Fold 2 IBS: 0.23230185713276866
Fold 3 IBS: 0.22972040438907473
Fold 4 IBS: 0.2411735138843142
Fold 5 IBS: 0.23026842457986174
[I 2024-04-15 17:34:40,955] Trial 48 finished with value: 0.2360881017346947 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 2, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.33846670341980367, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.39308816597832114}. Best is trial 46 with value: 0.20370576584430547.
Fold 1 IBS: 0.2259877395146257
Fold 2 IBS: 0.19108809720640413
Fold 3 IBS: 0.20774889959066922
Fold 4 IBS: 0.2003727390941353
Fold 5 IBS: 0.20342585963506513
[I 2024-04-15 17:34:43,742] Trial 49 finished with value: 0.2057246670081799 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 19, 'min_samples_leaf': 5, 'max_depth': 1, 'n_estimators': 194, 'oob_score': True, 'max_samples': 0.2725130585270163, 'max_features': 'auto', 'min_weight_fraction_leaf

Fold 5 IBS: 0.20917003156588318
[I 2024-04-15 17:35:30,402] Trial 63 finished with value: 0.2184606447500307 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 220, 'oob_score': True, 'max_samples': 0.2325243517835534, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.11132771366310104}. Best is trial 46 with value: 0.20370576584430547.
Fold 1 IBS: 0.2308036001809005
Fold 2 IBS: 0.19285062563411273
Fold 3 IBS: 0.20338231310806063
Fold 4 IBS: 0.20244527646462443
Fold 5 IBS: 0.20372242165671908
[I 2024-04-15 17:35:33,187] Trial 64 finished with value: 0.2066408474088835 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 286, 'oob_score': True, 'max_samples': 0.1715834543920205, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0022361848547431257}. Best is trial 46 with value: 0.20370576584430547.
Fold 1 IBS: 0.24611925529321832
Fold 2 IBS: 0.23

Fold 2 IBS: 0.181186571358148
Fold 3 IBS: 0.21658784650405408
Fold 4 IBS: 0.19184268124137188
Fold 5 IBS: 0.1925984231888799
[I 2024-04-15 17:36:13,106] Trial 79 finished with value: 0.20079474200950248 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 14, 'n_estimators': 25, 'oob_score': True, 'max_samples': 0.658103911048427, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.1264213492372395}. Best is trial 79 with value: 0.20079474200950248.
Fold 1 IBS: 0.14621065223036303
Fold 2 IBS: 0.21835543795889995
Fold 3 IBS: 0.2614037742945736
Fold 4 IBS: 0.22042199270046206
Fold 5 IBS: 0.2424449916206397
[I 2024-04-15 17:36:13,313] Trial 80 finished with value: 0.2177673697609877 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 10, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 2, 'oob_score': True, 'max_samples': 0.6070678390359827, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.13087324504203068}. Best is tri

Fold 1 IBS: 0.23035635370923183
Fold 2 IBS: 0.19074321594876992
Fold 3 IBS: 0.2194527294610163
Fold 4 IBS: 0.18915063569781565
Fold 5 IBS: 0.19826232609117034
[I 2024-04-15 17:36:28,186] Trial 95 finished with value: 0.20559305218160082 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 139, 'oob_score': True, 'max_samples': 0.8075223534134511, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.12608769469565542}. Best is trial 79 with value: 0.20079474200950248.
Fold 1 IBS: 0.23556491153783007
Fold 2 IBS: 0.19006041623205322
Fold 3 IBS: 0.2223584908159212
Fold 4 IBS: 0.1864963500371157
Fold 5 IBS: 0.197686962784186
[I 2024-04-15 17:36:30,070] Trial 96 finished with value: 0.20643342628142122 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 182, 'oob_score': True, 'max_samples': 0.5972550179693354, 'max_features': 'auto', 'min_weight_fraction_leaf

In [136]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [137]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.782
train_ibs:  0.201


#### Test

In [138]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [139]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=11, max_features='log2', max_leaf_nodes=14,
                     max_samples=0.9805420780206322, min_samples_leaf=16,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.2725745032335037,
                     n_estimators=349, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.512


RandomSurvivalForest(max_depth=14, max_features='auto', max_leaf_nodes=10,
                     max_samples=0.658103911048427, min_samples_leaf=2,
                     min_samples_split=17,
                     min_weight_fraction_leaf=0.1264213492372395,
                     n_estimators=25, oob_score=True, random_state=123)

test_ibs:  0.256


In [140]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [141]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [142]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 17:36:36,843] A new study created in memory with name: no-name-a568dac9-a2f7-4405-bdbd-8851879ad937


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.7945736434108527
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.8250950570342205
Fold 5 C-index: 0.7854077253218884
[I 2024-04-15 17:36:37,581] Trial 0 finished with value: 0.7560648740319208 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7560648740319208.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:36:39,577] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.7887596899224806
Fold 3 C-index: 0.6936170212765957
Fold 4 C-index: 0.7547528517110266
Fold 5 C-index: 0.7682403433476395
[I 2024-04-15 17:37:00,122] Trial 16 finished with value: 0.7166118298571261 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7574661942157834.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.8062015503875969
Fold 3 C-index: 0.7574468085106383
Fold 4 C-index: 0.7908745247148289
Fold 5 C-index: 0.7725321888412017
[I 2024-04-15 17:37:00,766] Trial 17 finished with value: 0.7513074288334827 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_feat

Fold 1 C-index: 0.6334661354581673
Fold 2 C-index: 0.8062015503875969
Fold 3 C-index: 0.7489361702127659
Fold 4 C-index: 0.7946768060836502
Fold 5 C-index: 0.7725321888412017
[I 2024-04-15 17:37:15,956] Trial 31 finished with value: 0.7511625701966765 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 500, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9462932212483206, 'min_weight_fraction_leaf': 0.2256775992503665}. Best is trial 12 with value: 0.7574661942157834.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7984496124031008
Fold 3 C-index: 0.7617021276595745
Fold 4 C-index: 0.7984790874524715
Fold 5 C-index: 0.7811158798283262
[I 2024-04-15 17:37:16,849] Trial 32 finished with value: 0.7522521303133161 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features

...

Fold 1 IBS: 0.24516664882076875
Fold 2 IBS: 0.23125368314974099
Fold 3 IBS: 0.2293568925972559
Fold 4 IBS: 0.24026892015315723
Fold 5 IBS: 0.2284688763963705
[I 2024-04-15 17:40:57,764] Trial 30 finished with value: 0.2349030042234587 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 408, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6124736134433101, 'min_weight_fraction_leaf': 0.2917886569515737}. Best is trial 23 with value: 0.21058740602327014.
Fold 1 IBS: 0.23047082448431727
Fold 2 IBS: 0.20246891396287392
Fold 3 IBS: 0.21289749096154267
Fold 4 IBS: 0.20606546111822555
Fold 5 IBS: 0.20284535946507126
[I 2024-04-15 17:41:01,248] Trial 31 finished with value: 0.21094960999840615 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 461, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.83

Fold 1 IBS: 0.23464572254213412
Fold 2 IBS: 0.19994116107207244
Fold 3 IBS: 0.2110451082896479
Fold 4 IBS: 0.20480646084800908
Fold 5 IBS: 0.19826938161562588
[I 2024-04-15 17:41:50,368] Trial 45 finished with value: 0.20974156687349788 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 7, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 144, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9550453481916195, 'min_weight_fraction_leaf': 0.0043820704059355925}. Best is trial 43 with value: 0.207160013172743.
Fold 1 IBS: 0.2352188360353779
Fold 2 IBS: 0.21012288416075062
Fold 3 IBS: 0.22006135263039905
Fold 4 IBS: 0.21420786948504866
Fold 5 IBS: 0.2046943511292496
[I 2024-04-15 17:41:51,759] Trial 46 finished with value: 0.2168610586881652 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 10, 'n_estimators': 138, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.956

Fold 1 IBS: 0.23661847734506994
Fold 2 IBS: 0.22366715614035915
Fold 3 IBS: 0.2242603393930627
Fold 4 IBS: 0.22833423638241543
Fold 5 IBS: 0.2179281861428918
[I 2024-04-15 17:42:28,926] Trial 60 finished with value: 0.22616167908075982 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 17, 'n_estimators': 296, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.9882475281225217, 'min_weight_fraction_leaf': 0.2583710069782479}. Best is trial 55 with value: 0.2060207484941457.
Fold 1 IBS: 0.2293695725263114
Fold 2 IBS: 0.20363219479344702
Fold 3 IBS: 0.2088236286171769
Fold 4 IBS: 0.1987984694721575
Fold 5 IBS: 0.19493278055546723
[I 2024-04-15 17:42:32,002] Trial 61 finished with value: 0.207111329192912 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 354, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.94882

Fold 1 IBS: 0.23880493280560305
Fold 2 IBS: 0.22211084601176734
Fold 3 IBS: 0.22536872182721576
Fold 4 IBS: 0.22893705950842794
Fold 5 IBS: 0.21910939176460062
[I 2024-04-15 17:43:05,285] Trial 75 finished with value: 0.2268661903835229 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 14, 'n_estimators': 288, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.23212254840913404, 'min_weight_fraction_leaf': 0.06322380085514831}. Best is trial 64 with value: 0.2057071749878002.
Fold 1 IBS: 0.24495500676299287
Fold 2 IBS: 0.23109350863163353
Fold 3 IBS: 0.22939694619359435
Fold 4 IBS: 0.24024660193684408
Fold 5 IBS: 0.2282876800438737
[I 2024-04-15 17:43:08,582] Trial 76 finished with value: 0.2347959487137877 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 20, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 339, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples':

Fold 1 IBS: 0.2288522415361709
Fold 2 IBS: 0.20354231400911754
Fold 3 IBS: 0.21065867348158182
Fold 4 IBS: 0.19846433877057512
Fold 5 IBS: 0.19526700958232354
[I 2024-04-15 17:43:47,455] Trial 90 finished with value: 0.20735691547595375 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 373, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.9849096047855805, 'min_weight_fraction_leaf': 0.028193081489496362}. Best is trial 64 with value: 0.2057071749878002.
Fold 1 IBS: 0.23693605369947957
Fold 2 IBS: 0.2044148977534312
Fold 3 IBS: 0.20934155605387997
Fold 4 IBS: 0.19204742285057097
Fold 5 IBS: 0.19182732197423547
[I 2024-04-15 17:43:50,221] Trial 91 finished with value: 0.20691345046631943 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 13, 'n_estimators': 303, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples':

In [143]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [144]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.206


#### Test

In [145]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [146]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=18, max_leaf_nodes=14,
                   max_samples=0.6321905190335297, min_samples_leaf=1,
                   min_samples_split=8,
                   min_weight_fraction_leaf=0.009478700458801217,
                   n_estimators=101, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.485


ExtraSurvivalTrees(max_depth=15, max_features='auto', max_leaf_nodes=20,
                   max_samples=0.9183889354438057, min_samples_leaf=1,
                   min_samples_split=18,
                   min_weight_fraction_leaf=0.0017390275642744837,
                   n_estimators=313, random_state=123, warm_start=True)

IBS: 0.254


In [147]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [148]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-15 17:44:16,212] A new study created in memory with name: no-name-1feb84ca-550b-4b6d-a826-dee457c6bc66


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:44:34,479] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:44:43,920] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:49:23,408] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.70205736041358.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:50:01,501] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squared_

Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:57:16,846] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 9 with value: 0.70205736041358.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-15 17:57:32,987] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedman_mse', 'ccp_alpha': 4.582122091349964, 'm

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 18:33:13,916] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.918589848048704, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.1366452847323028, 'n_estimators': 388, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.18967222528443176, 'max_features': 'auto', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 19, 'max_depth': 18}. Best is trial 22 with value: 0.23475160318907085.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:33:40,952] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7513175598857118, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927115
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:37:23,412] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8952643616873973, 'learning_rate': 0.05359198006915804, 'dropout_rate': 0.6509686174553241, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.713342732410399, 'min_weight_fraction_leaf': 0.037327349410482574, 'max_features': 'auto', 'min_impurity_decrease': 2.2946753237767036e-06, 'validation_fraction': 0.8670144195682054, 'min_samples_split': 9, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 18}. Best is trial 22 with value: 0.23475160318907085.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:37:35,651] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9559398584951578, 'learning_rate': 0.0013120255

Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-15 18:41:05,820] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9040601608260395, 'learning_rate': 0.008028762030775153, 'dropout_rate': 0.1661055390191164, 'n_estimators': 412, 'criterion': 'squared_error', 'ccp_alpha': 0.5920167940309409, 'min_weight_fraction_leaf': 0.20760648939509768, 'max_features': 1, 'min_impurity_decrease': 1.2858411523836384e-06, 'validation_fraction': 0.7519570151291436, 'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 22 with value: 0.23475160318907085.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:41:17,822] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9412481314247185, 'learning_rate': 0.003888190949209

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:44:56,051] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8783477148356468, 'learning_rate': 0.019145642246983192, 'dropout_rate': 0.2500967923284591, 'n_estimators': 479, 'criterion': 'squared_error', 'ccp_alpha': 1.2742275973203492, 'min_weight_fraction_leaf': 0.11068414611763369, 'max_features': 'auto', 'min_impurity_decrease': 0.0011701047770450368, 'validation_fraction': 0.9552938036430088, 'min_samples_split': 14, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 4}. Best is trial 85 with value: 0.23282798427356788.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:45:26,700] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9225585795675643, 'learning_rate': 0.0028206049

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-15 18:49:51,313] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9084360842440081, 'learning_rate': 0.013060448266216875, 'dropout_rate': 0.25443633448028735, 'n_estimators': 468, 'criterion': 'squared_error', 'ccp_alpha': 0.2223066295281712, 'min_weight_fraction_leaf': 0.25005462596245476, 'max_features': 'auto', 'min_impurity_decrease': 2.1305992007975229e-07, 'validation_fraction': 0.4189919199463679, 'min_samples_split': 15, 'max_leaf_nodes': 12, 'min_samples_leaf': 19, 'max_depth': 1}. Best is trial 85 with value: 0.23282798427356788.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.23282798427356788], datetime_start=datetime.datetime(2024, 4, 15, 18, 43, 11, 876328), datetime_complete=datetime.datetime(2024, 4, 15, 18, 43, 40, 12891), params={'subsample': 0.9481727373988897,

In [149]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [150]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.729
train_ibs:  0.233


#### Test

In [151]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [152]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0048717552527591836,
                                 dropout_rate=0.91502503958971,
                                 learning_rate=0.03769488430329127, max_depth=6,
                                 max_features=0.1, max_leaf_nodes=11,
                                 min_impurity_decrease=0.09932930546575598,
                                 min_samples_leaf=12, min_samples_split=18,
                                 min_weight_fraction_leaf=0.3032600022438751,
                                 n_estimators=151, random_state=123,
                                 subsample=0.5113292744810009,
                                 validation_fraction=0.7524856089234744)

C-index score: 0.499


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01138981446692314,
                                 criterion='squared_error',
                                 dropout_rate=0.2479619862207481,
                                 learning_rate=0.02022975259096714, max_depth=8,
                                 max_features='auto', max_leaf_nodes=20,
                                 min_impurity_decrease=9.61920586779085e-07,
                                 min_samples_leaf=18, min_samples_split=14,
                                 min_weight_fraction_leaf=0.026767353450782638,
                                 n_estimators=438, random_state=123,
                                 subsample=0.9481727373988897,
                                 validation_fraction=0.973754058089352)

IBS: 0.229


In [153]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [154]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [155]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-15 18:49:57,943] A new study created in memory with name: no-name-ccbd4d30-ef76-43fe-b979-ea736700f5e7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 18:49:58,485] Trial 0 finished with value: 0.6423526693826518 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6423526693826518.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 18:50:02,504] Trial 1 finished with value: 0.6456489898043869 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6456489898043869.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6653992395437263
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 18:50:28,655] Trial 19 finished with value: 0.6436924938049693 and parameters: {'subsample': 0.403731145647275, 'dropout_rate': 0.9947981608913361, 'n_estimators': 400, 'learning_rate': 0.04269105711861253}. Best is trial 16 with value: 0.657004876911789.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-15 18:50:29,739] Trial 20 finished with value: 0.6471335458766757 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.7933084651006226, 'n_estimators': 258, 'learning_rate': 0.023714096641497005}. Best is trial 16 with value: 0.657004876911789.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7170542635658915
Fold 3 C-index: 0.5617021276595745
Fol

Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7093023255813954
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6844106463878327
Fold 5 C-index: 0.6738197424892703
[I 2024-04-15 18:50:49,203] Trial 38 finished with value: 0.6461656935232163 and parameters: {'subsample': 0.30366897196288706, 'dropout_rate': 0.13272164755980653, 'n_estimators': 35, 'learning_rate': 0.04918505031160733}. Best is trial 27 with value: 0.6699472340896035.
Fold 1 C-index: 0.601593625498008
Fold 2 C-index: 0.7054263565891473
Fold 3 C-index: 0.5574468085106383
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6523605150214592
[I 2024-04-15 18:50:50,626] Trial 39 finished with value: 0.6410080466751813 and parameters: {'subsample': 0.23052838497416736, 'dropout_rate': 0.8890898259545601, 'n_estimators': 227, 'learning_rate': 0.0605670628016505}. Best is trial 27 with value: 0.6699472340896035.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fol

...

Fold 5 IBS: 0.22885900132307044
[I 2024-04-15 18:52:00,349] Trial 19 finished with value: 0.23099360328696633 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.22813963029627435
Fold 2 IBS: 0.21261919492535192
Fold 3 IBS: 0.2328032377684259
Fold 4 IBS: 0.23177471930860935
Fold 5 IBS: 0.21241390693232276
[I 2024-04-15 18:52:00,669] Trial 20 finished with value: 0.22355013784619687 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.23294882482388735
Fold 2 IBS: 0.2172731528360657
Fold 3 IBS: 0.23037554967791973
Fold 4 IBS: 0.23393962524579212
Fold 5 IBS: 0.21554329357937044
[I 2024-04-15 18:52:00,976] Trial 21 finished with value: 0.22601608923260708 and parameters: {'subsamp

Fold 1 IBS: 0.22935008117918856
Fold 2 IBS: 0.21153618549256686
Fold 3 IBS: 0.29072451950301204
Fold 4 IBS: 0.2586656669929489
Fold 5 IBS: 0.24321840787686647
[I 2024-04-15 18:52:07,797] Trial 39 finished with value: 0.24669897220891657 and parameters: {'subsample': 0.7076071631229807, 'dropout_rate': 0.36113352605756727, 'n_estimators': 75, 'learning_rate': 0.05598015324329953}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.4295827306814518
Fold 2 IBS: 0.33729766730917193
Fold 3 IBS: 0.38924638966253877
Fold 4 IBS: 0.31325291352447
Fold 5 IBS: 0.32175848063875884
[I 2024-04-15 18:52:11,340] Trial 40 finished with value: 0.35822763636327826 and parameters: {'subsample': 0.8123581601061346, 'dropout_rate': 0.27073406810212547, 'n_estimators': 434, 'learning_rate': 0.050770250632636425}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.21843232747192648
Fold 2 IBS: 0.20124696623375102
Fold 3 IBS: 0.26589789731233937
Fold 4 IBS: 0.23953030936979589
Fold 5 

Fold 5 IBS: 0.2104012829571286
[I 2024-04-15 18:52:20,336] Trial 58 finished with value: 0.22130143678183142 and parameters: {'subsample': 0.5793765216559574, 'dropout_rate': 0.869359581862351, 'n_estimators': 38, 'learning_rate': 0.03754893539001649}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22332388143407295
Fold 2 IBS: 0.20363163283585628
Fold 3 IBS: 0.24449748198927226
Fold 4 IBS: 0.22907918428885007
Fold 5 IBS: 0.21077017722074096
[I 2024-04-15 18:52:20,572] Trial 59 finished with value: 0.22226047155375853 and parameters: {'subsample': 0.4535266512643318, 'dropout_rate': 0.9611194785854935, 'n_estimators': 40, 'learning_rate': 0.038908657923000085}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.2426474010909224
Fold 2 IBS: 0.22436959888835237
Fold 3 IBS: 0.3051332800511575
Fold 4 IBS: 0.26525378607334044
Fold 5 IBS: 0.25799073524428723
[I 2024-04-15 18:52:21,140] Trial 60 finished with value: 0.25907896026961197 and parameters: {'subsample

Fold 2 IBS: 0.20591143440013582
Fold 3 IBS: 0.280976248290565
Fold 4 IBS: 0.24848562445786387
Fold 5 IBS: 0.23615081501710733
[I 2024-04-15 18:52:28,090] Trial 78 finished with value: 0.2390363508032336 and parameters: {'subsample': 0.6733814576309736, 'dropout_rate': 0.7358591929789065, 'n_estimators': 128, 'learning_rate': 0.02725879214599771}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22811684389438022
Fold 2 IBS: 0.21004117151133037
Fold 3 IBS: 0.23552715888942047
Fold 4 IBS: 0.22642754442179203
Fold 5 IBS: 0.2109726578133541
[I 2024-04-15 18:52:28,432] Trial 79 finished with value: 0.22221707530605544 and parameters: {'subsample': 0.4590324206086437, 'dropout_rate': 0.5730916522393441, 'n_estimators': 62, 'learning_rate': 0.016584715438210443}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.21914807252926094
Fold 2 IBS: 0.20152400712862856
Fold 3 IBS: 0.2680973864145784
Fold 4 IBS: 0.2372703504251833
Fold 5 IBS: 0.22464173105960364
[I 2024-04

Fold 5 IBS: 0.22220093036998428
[I 2024-04-15 18:52:35,920] Trial 97 finished with value: 0.22814309741112476 and parameters: {'subsample': 0.531829760385333, 'dropout_rate': 0.6847533776104763, 'n_estimators': 79, 'learning_rate': 0.03223905987322316}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.2466414230044532
Fold 2 IBS: 0.23146700010719856
Fold 3 IBS: 0.2289119634694628
Fold 4 IBS: 0.24164291053866815
Fold 5 IBS: 0.2288108925224838
[I 2024-04-15 18:52:36,104] Trial 98 finished with value: 0.2354948379284533 and parameters: {'subsample': 0.5618605706633095, 'dropout_rate': 0.2002801186536965, 'n_estimators': 1, 'learning_rate': 0.022985459193691663}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.21844021664292554
Fold 2 IBS: 0.2014998540928507
Fold 3 IBS: 0.25108301890771456
Fold 4 IBS: 0.2341234904146111
Fold 5 IBS: 0.21405157380858064
[I 2024-04-15 18:52:36,520] Trial 99 finished with value: 0.2238396307733365 and parameters: {'subsample': 0.75

In [156]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [157]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.67
train_ibs:  0.221


#### Test

In [158]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [159]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6979039748471498,
                                              learning_rate=0.030307532796347948,
                                              n_estimators=1, random_state=123,
                                              subsample=0.19718036296309377)

C-index score: 0.521


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8767395889398294,
                                              learning_rate=0.030725485144314495,
                                              n_estimators=48, random_state=123,
                                              subsample=0.5967091670188371)

IBS: 0.239


In [160]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [161]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.791,1.0
Randomsurvivalforest,0.782,2.0
GradientBoosting,0.729,3.0
CoxElastic,0.677,4.0
CoxPH,0.675,5.0
CoxLasso,0.673,6.0
ComponentwiseGradientBoosting,0.670,7.0
CoxRidge,0.649,8.0


In [162]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.201,1.0
ExtraSurvivalTrees,0.206,2.0
CoxLasso,0.220,3.5
CoxElastic,0.220,3.5
CoxPH,0.221,5.5
ComponentwiseGradientBoosting,0.221,5.5
GradientBoosting,0.233,7.0
CoxRidge,0.236,8.0


In [163]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.534,1.0
ComponentwiseGradientBoosting,0.521,2.0
Randomsurvivalforest,0.512,3.0
GradientBoosting,0.499,4.0
CoxPH,0.498,5.0
CoxLasso,0.497,6.5
CoxElastic,0.497,6.5
ExtraSurvivalTrees,0.485,8.0


In [164]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
CoxRidge,0.229,1.5
GradientBoosting,0.229,1.5
ComponentwiseGradientBoosting,0.239,3.0
ExtraSurvivalTrees,0.254,4.0
Randomsurvivalforest,0.256,5.0
CoxLasso,0.279,6.0
CoxElastic,0.280,7.0
CoxPH,0.281,8.0


In [165]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/dfs/yeojohnson/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_dfs_yeojohnson_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [166]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-15
